In [1]:
# Starting of phase 1 that is Data Cleaning - preparing the dataset for analysis

In [2]:
# Phase 1 - Step 2: Load the Dataset
import pandas as pd
import numpy as np

df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')

# First look at the data
print("Shape of dataset:", df.shape)
print("\nFirst 5 rows:")
df.head()


Shape of dataset: (7043, 21)

First 5 rows:


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
# Phase 1 - Step 3: Check Columns and Missing Values
print("Column names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values per column:")
print(df.isnull().sum())

Column names:
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']

Data types:
customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

Missing values per column:
customerID         

In [4]:
# Phase 1 - Step 4: Data Cleaning

# 1. Handle missing values in TotalCharges
print("Before cleaning - TotalCharges missing/blanks:", df['TotalCharges'].eq(' ').sum())

# Replace blank spaces with NaN
df['TotalCharges'] = df['TotalCharges'].replace(' ', np.nan)

# Convert to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

print("After cleaning - Missing values in TotalCharges:", df['TotalCharges'].isnull().sum())

# Fill missing TotalCharges with median (by contract type - more accurate)
df['TotalCharges'] = df.groupby('Contract')['TotalCharges'].transform(lambda x: x.fillna(x.median()))

print("\n✅ TotalCharges cleaned and converted to numeric!")
print(df['TotalCharges'].describe())

Before cleaning - TotalCharges missing/blanks: 11
After cleaning - Missing values in TotalCharges: 11

✅ TotalCharges cleaned and converted to numeric!
count    7043.000000
mean     2285.257099
std      2265.567157
min        18.800000
25%       402.225000
50%      1400.550000
75%      3786.600000
max      8684.800000
Name: TotalCharges, dtype: float64


In [5]:
# Phase 1 - Step 5: Cleaning & Feature Engineering (Final, Corrected)

# 1. Convert Yes/No columns to 1/0 (binary encoding) — done ONCE, correctly
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'MultipleLines', 
               'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
               'TechSupport', 'StreamingTV', 'StreamingMovies', 
               'PaperlessBilling', 'Churn']

for col in binary_cols:
    df[col] = df[col].map({
        'Yes': 1, 
        'No': 0, 
        'No internet service': 0, 
        'No phone service': 0
    })

print("✅ Binary columns encoded (Yes/No → 1/0)")
print(df[binary_cols].isnull().sum())  # sanity check: should all be 0

# 2. Fix TotalCharges missing values (grouped median by Contract type)
df['TotalCharges'] = df.groupby('Contract')['TotalCharges'].transform(
    lambda x: x.fillna(x.median())
)

# 3. Clean any remaining object columns (stray whitespace, etc.)
object_cols = df.select_dtypes(include=['object']).columns
for col in object_cols:
    df[col] = df[col].replace(' ', np.nan)

# 4. Create new useful features
df['AvgMonthlyCharge'] = df['TotalCharges'] / df['tenure'].replace(0, 1)  # avoid division by zero
df['TenureBand'] = pd.cut(df['tenure'], bins=[0, 12, 24, 48, 72, 100], 
                          labels=['0-12', '13-24', '25-48', '49-72', '73+'])

print("\n✅ New features created:")
print(df[['tenure', 'TotalCharges', 'AvgMonthlyCharge', 'TenureBand']].head())

# 5. Final missing values check
print("\nRemaining missing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# 6. Confirm Churn is correctly encoded before saving
print("\nChurn value check:")
print(df['Churn'].unique())
print(df['Churn'].value_counts())

# 7. Save the cleaned dataset
df.to_csv('../data/telco_cleaned.csv', index=False)
print("\n✅ Cleaned dataset saved as 'telco_cleaned.csv' in data folder!")

✅ Binary columns encoded (Yes/No → 1/0)
Partner             0
Dependents          0
PhoneService        0
MultipleLines       0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
PaperlessBilling    0
Churn               0
dtype: int64

✅ New features created:
   tenure  TotalCharges  AvgMonthlyCharge TenureBand
0       1         29.85         29.850000       0-12
1      34       1889.50         55.573529      25-48
2       2        108.15         54.075000       0-12
3      45       1840.75         40.905556      25-48
4       2        151.65         75.825000       0-12

Remaining missing values:
TenureBand    11
dtype: int64

Churn value check:
[0 1]
Churn
0    5174
1    1869
Name: count, dtype: int64

✅ Cleaned dataset saved as 'telco_cleaned.csv' in data folder!


C:\Users\hp\AppData\Local\Temp\ipykernel_41296\4074045849.py:26: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = df.select_dtypes(include=['object']).columns


In [6]:
df['TenureBand'] = pd.cut(df['tenure'], bins=[0, 12, 24, 48, 72, 100], 
                          labels=['0-12', '13-24', '25-48', '49-72', '73+'],
                          include_lowest=True)

print(df['TenureBand'].isnull().sum())  # should be 0 now

df.to_csv('../data/telco_cleaned.csv', index=False)
print("✅ Cleaned dataset re-saved with TenureBand fix")

0
✅ Cleaned dataset re-saved with TenureBand fix


In [7]:
# Phase 1 - Step 8: Fix TenureBand and Complete Data Cleaning

# Fix TenureBand missing values
print("Before fix - Missing in TenureBand:", df['TenureBand'].isnull().sum())

# Fill missing TenureBand
df['TenureBand'] = df['TenureBand'].fillna('0-12')

# Final verification
print("After fix - Missing in TenureBand:", df['TenureBand'].isnull().sum())
print("\nTotal remaining missing values in entire dataset:", df.isnull().sum().sum())

# Final data quality check
print("\n✅ Final Data Quality Summary:")
print("Total rows:", len(df))
print("Total columns:", len(df.columns))
print("Missing values:", df.isnull().sum().sum())
print("\nData types:")
print(df.dtypes.value_counts())

# Save the final cleaned version
df.to_csv('../data/telco_cleaned_final.csv', index=False)
print("\n🎉 Phase 1 Completed! Cleaned dataset is ready.")

Before fix - Missing in TenureBand: 0
After fix - Missing in TenureBand: 0

Total remaining missing values in entire dataset: 0

✅ Final Data Quality Summary:
Total rows: 7043
Total columns: 23
Missing values: 0

Data types:
int64       14
str          5
float64      3
category     1
Name: count, dtype: int64

🎉 Phase 1 Completed! Cleaned dataset is ready.


In [8]:
# Phase 1 - Final Step: Drop Unnecessary Columns

print("Original shape:", df.shape)

# Drop columns we don't need
columns_to_drop = ['customerID']

df_clean = df.drop(columns=columns_to_drop, errors='ignore')

print("After dropping unnecessary columns:")
print("New shape:", df_clean.shape)
print("Remaining columns:", df_clean.columns.tolist())

# Save the final slimmed version
df_clean.to_csv('../data/telco_cleaned_final.csv', index=False)
print("\n✅ Slimmed dataset saved as 'telco_cleaned_final.csv'")

# Update df to use the cleaned version going forward
df = df_clean.copy()

Original shape: (7043, 23)
After dropping unnecessary columns:
New shape: (7043, 22)
Remaining columns: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn', 'AvgMonthlyCharge', 'TenureBand']

✅ Slimmed dataset saved as 'telco_cleaned_final.csv'


In [9]:
print(df.isnull().sum()[df.isnull().sum() > 0])

Series([], dtype: int64)


In [10]:
df[df.isnull().any(axis=1)]

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,AvgMonthlyCharge,TenureBand


In [11]:
# Fill missing TotalCharges with median
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

# Verify again
print("Remaining missing values:", df.isnull().sum().sum())

Remaining missing values: 0


In [12]:
df_final_check = pd.read_csv('../data/telco_cleaned_final.csv')

print("Shape:", df_final_check.shape)
print("\nChurn check:")
print(df_final_check['Churn'].unique())
print(df_final_check['Churn'].value_counts())
print("\nMissing values:", df_final_check.isnull().sum().sum())
print("\nColumns:", list(df_final_check.columns))

Shape: (7043, 22)

Churn check:
[0 1]
Churn
0    5174
1    1869
Name: count, dtype: int64

Missing values: 0

Columns: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn', 'AvgMonthlyCharge', 'TenureBand']


In [13]:
# Check duplicates BEFORE dropping customerID
df_raw_check = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
print("Duplicate customerIDs:", df_raw_check['customerID'].duplicated().sum())
print("Duplicate full rows (excluding customerID):", df_raw_check.drop(columns=['customerID']).duplicated().sum())

Duplicate customerIDs: 0
Duplicate full rows (excluding customerID): 22


In [14]:
df_final_check = pd.read_csv('../data/telco_cleaned_final.csv')
print("Shape:", df_final_check.shape)
print(df_final_check['Churn'].value_counts())
print("Missing values:", df_final_check.isnull().sum().sum())

Shape: (7043, 22)
Churn
0    5174
1    1869
Name: count, dtype: int64
Missing values: 0


In [15]:
import json

with open('01_data_cleaning.ipynb', 'r', encoding='utf-8') as f:
    nb = json.load(f)

for i, cell in enumerate(nb['cells']):
    source = ''.join(cell.get('source', []))
    if 'duplicate' in source.lower():
        print(f"--- Cell {i} ---")
        print(source)
        print()

--- Cell 12 ---
# Check duplicates BEFORE dropping customerID
df_raw_check = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
print("Duplicate customerIDs:", df_raw_check['customerID'].duplicated().sum())
print("Duplicate full rows (excluding customerID):", df_raw_check.drop(columns=['customerID']).duplicated().sum())

--- Cell 14 ---
import json

with open('01_data_cleaning.ipynb', 'r', encoding='utf-8') as f:
    nb = json.load(f)

for i, cell in enumerate(nb['cells']):
    source = ''.join(cell.get('source', []))
    if 'duplicate' in source.lower():
        print(f"--- Cell {i} ---")
        print(source)
        print()



In [16]:
df_final_check = pd.read_csv('../data/telco_cleaned_final.csv')
print("Shape:", df_final_check.shape)
print(df_final_check['Churn'].value_counts())
print("Missing values:", df_final_check.isnull().sum().sum())

Shape: (7043, 22)
Churn
0    5174
1    1869
Name: count, dtype: int64
Missing values: 0


In [17]:
df_check = pd.read_csv('../data/telco_cleaned.csv')
print(df_check.shape)
print(df_check['Churn'].value_counts())

(7043, 23)
Churn
0    5174
1    1869
Name: count, dtype: int64
